# Stacking e combinação de modelos

**Objetivo:** combinar três modelos diferentes por soft voting e por stacking, e comparar a acurácia da combinação com a de cada modelo isolado.

In [ ]:
# bibliotecas base
import numpy as np
import pandas as pd

# Plotly para os gráficos (interativos e leves no Colab)
import plotly.graph_objects as go
import plotly.express as px
import plotly.io as pio
pio.templates.default = "simple_white"

# paleta do curso (a mesma do site)
AZUL, VERMELHO, VERDE = "#3266ad", "#c0392b", "#1a7a4a"
TINTA, SUAVE = "#1c1e15", "#6b7050"

# reprodutibilidade: uma única semente para tudo que é aleatório
SEMENTE = 42
np.random.seed(SEMENTE)

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

dados = load_breast_cancer()
X, y = dados.data, dados.target
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3,
                                          random_state=SEMENTE, stratify=y)

## 1. Três modelos de base diferentes

Uma regressão logística, uma SVM (com probabilidade) e uma floresta. São modelos de naturezas distintas — a diversidade é o que faz a combinação valer a pena.

In [ ]:
base = {
    "Regressao logistica": make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000)),
    "SVM (RBF)":           make_pipeline(StandardScaler(), SVC(probability=True, random_state=SEMENTE)),
    "Floresta":            RandomForestClassifier(n_estimators=200, random_state=SEMENTE),
}
for nome, m in base.items():
    m.fit(X_tr, y_tr)
    print(nome.ljust(22), "acuracia:", round(m.score(X_te, y_te), 3))

## 2. Soft voting e stacking

O `VotingClassifier` (soft) faz a **média das probabilidades**. O `StackingClassifier` treina um **meta-modelo** (uma regressão logística) sobre as previsões dos três — usando previsões out-of-fold para não vazar.

In [ ]:
from sklearn.ensemble import VotingClassifier, StackingClassifier

estimadores = [
    ("lr", make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000))),
    ("svm", make_pipeline(StandardScaler(), SVC(probability=True, random_state=SEMENTE))),
    ("rf", RandomForestClassifier(n_estimators=200, random_state=SEMENTE)),
]

votacao = VotingClassifier(estimators=estimadores, voting="soft")
votacao.fit(X_tr, y_tr)
print("soft voting  -> acuracia:", round(votacao.score(X_te, y_te), 3))

empilhado = StackingClassifier(estimators=estimadores,
                               final_estimator=LogisticRegression(max_iter=5000), cv=5)
empilhado.fit(X_tr, y_tr)
print("stacking     -> acuracia:", round(empilhado.score(X_te, y_te), 3))

## 3. Comparação final

Colocamos tudo lado a lado — e o resultado é honesto: quando um modelo de base já é muito forte (aqui, a regressão logística), a combinação fica **competitiva, mas pode não superá-lo**. O ganho do stacking aparece quando **nenhum** modelo domina sozinho; misturar não é mágica.

In [ ]:
nomes = []
valores = []
for nome, m in base.items():
    nomes.append(nome); valores.append(m.score(X_te, y_te))
nomes += ["Soft voting", "Stacking"]
valores += [votacao.score(X_te, y_te), empilhado.score(X_te, y_te)]

cores = [SUAVE, SUAVE, SUAVE, AZUL, VERDE]
figura = go.Figure(go.Bar(x=valores, y=nomes, orientation="h", marker_color=cores,
                          text=[round(v, 3) for v in valores], textposition="outside"))
figura.update_layout(title="Modelos de base x combinacoes",
                     xaxis_title="acuracia no teste", xaxis_range=[0.9, 1.0],
                     height=360, margin=dict(l=10, r=10, t=50, b=10))
figura.show()

## Exercício

Se os três modelos de base tivessem exatamente a mesma acurácia **e** errassem sempre nos mesmos exemplos, o que aconteceria com o soft voting?

<details><summary>Ver resposta</summary>

O soft voting **não ganharia nada**: a média de probabilidades de modelos que erram nos mesmos casos continua errando nesses casos. O benefício da combinação vem da **diversidade** — modelos que falham em exemplos diferentes, de modo que a média cancela erros individuais. Sem diversidade, votar apenas reproduz os mesmos acertos e erros.

</details>